# Experiment 2 — hot and cold in contact

A 2D crystal of 1440 identical atoms. First a heat bath holds the left half at T* = 0.8 and the right half at T* = 0.2 (`fix ... nvt`). Then both heat baths are removed and the crystal evolves by Newton's equations alone (`fix 1 all nve`): the total energy is conserved and the two halves can only exchange heat with each other.

**Before you look at the plot, predict** the final temperature of each half.

In [ ]:
# lammps-logfile is served from Atomify's own package index (no network needed).
%pip install -q lammps-logfile pandas

In [ ]:
import glob, os, time
import numpy as np
import lammps_logfile
import matplotlib.pyplot as plt

# Atomify stores every run of this project in runs/<run-name>/ next to this
# notebook (input snapshot, log.lammps, dumps). Pick the newest run here;
# use logs[0], logs[1], ... to look at older ones.
logs = sorted(glob.glob("runs/*/log.lammps"))
if not logs:
    raise RuntimeError("No runs yet: press Run in Atomify, wait for it to finish, then re-run this cell.")
print("Runs found:", *logs, sep="\n  ")

for attempt in range(5):
    try:
        log = lammps_logfile.File(logs[-1])
        break
    except FileNotFoundError:
        # Atomify may still be copying the finished run into the project.
        time.sleep(1)
        os.listdir(os.path.dirname(logs[-1]))
else:
    raise RuntimeError(f"{logs[-1]} is not readable yet: wait for the run to finish, then re-run this cell.")
print("Log keywords:", log.get_keywords())

The two thermometers (`c_Tl`, `c_Tr` = the temperature of each half) against time. The dotted line marks the moment the heat baths are removed.

In [ ]:
t  = log.get("Time")
Tl = log.get("c_Tl")
Tr = log.get("c_Tr")
E  = log.get("TotEng")
t_contact = log.get("Time", run_num=0)[-1]      # end of the preparation run

plt.figure(figsize=(8, 4))
plt.plot(t, Tl, color="tab:red",  label="left half (starts hot)")
plt.plot(t, Tr, color="tab:blue", label="right half (starts cold)")
plt.axvline(t_contact, color="k", ls=":", label="heat baths removed")
plt.xlabel("time t*"); plt.ylabel("T*"); plt.legend(); plt.show()

after = t > t_contact
tail  = t > t[-1] - 20
print(f"at contact:  T_left = {Tl[after][0]:.3f}   T_right = {Tr[after][0]:.3f}   mean = {(Tl[after][0] + Tr[after][0]) / 2:.3f}")
print(f"at the end:  T_left = {Tl[tail].mean():.3f}   T_right = {Tr[tail].mean():.3f}")
print(f"total energy drift after contact: {E[-1] - E[after][0]:.2e}  (should be ~0: Newton only)")

Note down:
* Did the two halves end at the same temperature? Is it the mean of the two starting temperatures — and why would it be (or not)?
* Roughly how long, in time units, did it take for the difference to disappear? (Zoom in: `plt.xlim(t_contact, t_contact + 20)`.)
* Is this the same thing as the two Einstein crystals of exercise 35 sharing energy quanta? What did that model predict, and what does it *not* tell you?
* Write, in your own words: what is *thermal equilibrium*, and what is *temperature*? (Exam 2019, problem 1.2.)